In [ ]:
import s3fs
import os
import pandas as pd
import time
import numpy as np
import numpy as np
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.multiclass import unique_labels
from sklearn.metrics import classification_report
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.multiclass import unique_labels
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.utils.class_weight import compute_sample_weight
import joblib
import pandas as pd
from tqdm import tqdm
import os
from tqdm import tqdm
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import io
import json


# S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"] #  on récupère l'url du stockage s3

# # on peut désormais instancier le filesystem en précisant l'url du stockage S3

# fs = s3fs.S3FileSystem(endpoint_url= S3_ENDPOINT_URL)
import psutil
# Get total memory in bytes
total_memory = psutil.virtual_memory().total

# Convert bytes to GB (optional, for readability)
total_memory_in_gb = total_memory / (1024 ** 3)

print(f"Total RAM: {total_memory_in_gb:.2f} GB")


Total RAM: 31.43 GB


In [44]:
print("Variables d'environnement disponibles :", list(os.environ.keys()))

Variables d'environnement disponibles : ['ALLUSERSPROFILE', 'APPDATA', 'CHROME_CRASHPAD_PIPE_NAME', 'COMMONPROGRAMFILES', 'COMMONPROGRAMFILES(X86)', 'COMMONPROGRAMW6432', 'COMPUTERNAME', 'COMSPEC', 'DRIVERDATA', 'EFC_12092_1262719628', 'EFC_12092_1592913036', 'EFC_12092_2283032206', 'EFC_12092_2775293581', 'EFC_12092_3789132940', 'ELECTRON_NO_ATTACH_CONSOLE', 'ELECTRON_RUN_AS_NODE', 'FPS_BROWSER_APP_PROFILE_STRING', 'FPS_BROWSER_USER_PROFILE_STRING', 'HADOOP_HOME', 'HOMEDRIVE', 'HOMEPATH', 'JAVA_HOME', 'JPY_INTERRUPT_EVENT', 'LOCALAPPDATA', 'LOGONSERVER', 'NUMBER_OF_PROCESSORS', 'ONEDRIVE', 'ONLINESERVICES', 'OS', 'PATH', 'PATHEXT', 'PLATFORMCODE', 'PROCESSOR_ARCHITECTURE', 'PROCESSOR_IDENTIFIER', 'PROCESSOR_LEVEL', 'PROCESSOR_REVISION', 'PROGRAMDATA', 'PROGRAMFILES', 'PROGRAMFILES(X86)', 'PROGRAMW6432', 'PROMPT', 'PSMODULEPATH', 'PUBLIC', 'PYDEVD_IPYTHON_COMPATIBLE_DEBUGGING', 'PYENV', 'PYENV_HOME', 'PYENV_ROOT', 'PYTHONIOENCODING', 'PYTHONUNBUFFERED', 'PYTHON_FROZEN_MODULES', 'REGION

In [45]:
import numpy as np

def cosine_similarity_matrix(X: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between all rows of X.
    X: (n, d)
    Returns: (n, n)
    """
    X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)
    return X_norm @ X_norm.T


# Text cleaning

This section cleans the description of job offers. It especially removes indiosyncratic sentences in job offers ("Thales est le leader mondial de XXX").

In [32]:
df = pd.read_parquet("data/JOCAS/small/jocas_small_all_half_sirenised_embed_job_title.parquet")

In [36]:
## Test possible sur un sub sample
df_small = df.sample(frac=0.1, random_state=42).reset_index(drop=True)


In [39]:
df.shape

(804997, 18)

In [40]:
df_small.to_parquet("data/JOCAS/small/jocas_small_all_sampled.parquet")

In [41]:
from tqdm.auto import tqdm
import math
from collections import Counter
import pandas as pd

def build_drop_sets_by_firm_clustered(
    df: pd.DataFrame,
    text_col="description",
    firm_col="entreprise_nom",
    share_threshold=0.10,
    min_count=2,
    sim_threshold=0.88,
    max_keys_per_firm=5000,
    max_offers_per_firm=None,
    random_state=0,
    remove_accents=True,
    collapse_digits=True,
    max_tokens=None,
    return_report=False,
    show_progress=True,
):
    """
    For each firm, identify sentence patterns that are repeated across many offers.

    Main idea:
    1. Split each offer into sentences.
    2. Normalize each sentence into a compact 'key'.
    3. Count how often each key appears across the firm's offers.
    4. Cluster very similar keys together (to merge near-duplicates).
    5. Mark clusters as boilerplate if they appear in enough offers.
    6. Return, for each firm, the set of sentence keys to remove later.
    """

    drop_by_firm = {}
    report = {}

    firms = df[firm_col].astype("string").fillna("").str.strip()
    grouped = firms.groupby(firms).groups.items()

    # Progress bar over firms
    if show_progress:
        grouped = tqdm(grouped, desc="Build drop sets (firms)")

    for firm, idx in grouped:
        if not firm:
            continue

        # Extract all texts for this firm
        sub = df.loc[idx, text_col].fillna("")

        # Optionally sample offers if a firm has too many
        if max_offers_per_firm is not None and len(sub) > max_offers_per_firm:
            sub = sub.sample(max_offers_per_firm, random_state=random_state)

        n_offers = len(sub)
        if n_offers == 0:
            if return_report:
                report[firm] = pd.DataFrame()
            continue

        offer_keys = []
        key_counts = Counter()

        # For each offer:
        # - split into sentences
        # - normalize each sentence into a key
        # - store unique keys seen in that offer
        for txt in sub:
            keys = {
                sentence_key(
                    s,
                    remove_accents=remove_accents,
                    collapse_digits=collapse_digits,
                    max_tokens=max_tokens,
                )
                for s in split_sentences(txt)
            }
            keys.discard("")  # remove empty keys
            offer_keys.append(keys)
            key_counts.update(keys)

        if not key_counts:
            if return_report:
                report[firm] = pd.DataFrame()
            continue

        # Keep only the most frequent keys if there are too many
        if len(key_counts) > max_keys_per_firm:
            top = {k for k, _ in key_counts.most_common(max_keys_per_firm)}
            key_counts = Counter({k: c for k, c in key_counts.items() if k in top})
            offer_keys = [ks & top for ks in offer_keys]

        # Sort keys by frequency, then cluster similar keys together
        keys_sorted = [k for k, _ in key_counts.most_common()]
        clusters = []
        key_to_cluster = {}

        for k in keys_sorted:
            assigned = False
            for ci, cl in enumerate(clusters):
                # If this key is similar enough to the cluster representative,
                # assign it to that cluster
                if key_sim(k, cl["rep"]) >= sim_threshold:
                    cl["members"].add(k)
                    key_to_cluster[k] = ci
                    assigned = True
                    break
            if not assigned:
                # Otherwise create a new cluster
                clusters.append({"rep": k, "members": {k}})
                key_to_cluster[k] = len(clusters) - 1

        # Compute document frequency at the cluster level:
        # in how many offers does each cluster appear?
        cluster_df = [0] * len(clusters)
        for ks in offer_keys:
            seen = set()
            for k in ks:
                ci = key_to_cluster.get(k)
                if ci is not None:
                    seen.add(ci)
            for ci in seen:
                cluster_df[ci] += 1

        # Threshold to decide whether a cluster is frequent enough to drop
        thresh = max(min_count, int(math.ceil(share_threshold * n_offers)))
        drop_clusters = {ci for ci, dfc in enumerate(cluster_df) if dfc >= thresh}

        # Convert drop clusters into a set of sentence keys to remove
        drop_keys = set()
        for ci in drop_clusters:
            drop_keys |= clusters[ci]["members"]

        if drop_keys:
            drop_by_firm[firm] = drop_keys

        # Optional report for inspection/debugging
        if return_report:
            rep = []
            for ci, cl in enumerate(clusters):
                rep.append({
                    "cluster_id": ci,
                    "df_offers": cluster_df[ci],         # number of offers containing this cluster
                    "share": cluster_df[ci] / n_offers, # share of offers containing it
                    "n_members": len(cl["members"]),     # number of similar keys in cluster
                    "rep": cl["rep"][:120],              # representative key
                })
            df_rep = pd.DataFrame(rep)
            if not df_rep.empty:
                df_rep = df_rep.sort_values(
                    ["df_offers", "n_members"], ascending=False
                )
            report[firm] = df_rep

    return (drop_by_firm, report) if return_report else drop_by_firm


# ----------------------------
# Apply removal (fast) with tqdm
# ----------------------------
def add_raw_clean_column(df: pd.DataFrame, text_col="description", out_col="_raw_clean", show_progress=True):
    """
    Pre-clean the raw text by removing HTML while keeping line structure,
    then replace line breaks with spaces.

    This creates an intermediate cleaned text column that can be reused later.
    """
    s = df[text_col].astype("string").fillna("")
    if show_progress:
        s = tqdm(s, desc="Pre-clean HTML", total=len(s))
    df[out_col] = s.map(clean_html_keep_newlines).str.replace("\n", " ", regex=False)
    return df


def remove_frequent_sentences_within_firm(
    df: pd.DataFrame,
    drop_by_firm: dict,
    text_col="description",
    firm_col="entreprise_nom",
    out_col="description_clean",
    raw_col="_raw_clean",
    min_keep_chars=200,
    remove_accents=True,
    collapse_digits=True,
    max_tokens=None,
    show_progress=True,
):
    """
    Remove firm-specific frequent/boilerplate sentences from each text.

    For each row:
    1. Split the original text into sentences.
    2. Normalize each sentence into a key.
    3. Drop the sentence if its key belongs to that firm's drop set.
    4. Rebuild the cleaned text and store it in out_col.
    """
    if raw_col not in df.columns:
        raise KeyError(f"Missing '{raw_col}'. Run add_raw_clean_column(df) first.")

    firms = df[firm_col].astype("string").fillna("").str.strip().values
    texts = df[text_col].fillna("").values
    raw_clean = df[raw_col].fillna("").values

    it = zip(texts, firms, raw_clean)
    if show_progress:
        it = tqdm(list(it), desc="Clean offers", total=len(df))

    cleaned = []
    for txt, firm, raw in it:
        # Split offer into sentences
        sents = split_sentences(txt)

        # Get the set of sentence keys to drop for this firm
        drops = drop_by_firm.get(firm, set())

        kept = []
        for sent in sents:
            k = sentence_key(
                sent,
                remove_accents=remove_accents,
                collapse_digits=collapse_digits,
                max_tokens=max_tokens
            )
            if k in drops:
                continue
            kept.append(sent)

        # Rebuild text with normalized spacing
        out = _RE_WS.sub(" ", " ".join(kept)).strip()
        cleaned.append(out)

    df[out_col] = cleaned
    return df

In [42]:
df["description"] = df["description"].str.lower()

# 1) apprendre quelles phrases sont "ultra fréquentes"
drop_by_firm = build_drop_sets_by_firm_clustered(
    df,
    text_col="description",
    firm_col="entreprise_nom",
    share_threshold=0.10,
)

# debug sur une firme:
# report["okaidi"].head(10)


Build drop sets (firms):   0%|          | 0/128358 [00:00<?, ?it/s]

NameError: name 'split_sentences' is not defined

In [26]:
df = add_raw_clean_column(
    df,
    text_col="description",
    out_col="_raw_clean",
    show_progress=False
)

NameError: name 'clean_html_keep_newlines' is not defined

In [ ]:
df = remove_frequent_sentences_within_firm(
    df,
    drop_by_firm,
    text_col="description",
    firm_col="entreprise_nom",
    out_col="description_clean",
    raw_col="_raw_clean",
    min_keep_chars=200,  # if you want, you can re-add fallback; see note below
    max_tokens=None,
    show_progress=True
)
df = df.drop(columns = "_raw_clean")


Clean offers: 100%|██████████| 81118/81118 [01:34<00:00, 855.67it/s]


In [ ]:
#df.to_parquet("jocas_small/jocas_small_all_half_sirenised.parquet")

In [ ]:
df_thales

,ID_JOCAS,annee,mois,entreprise_nom,location_zipcode,description,job_ROME_code,contractType,_raw_clean,description_clean
1130,cadremploi_2024-05-04_410,2024,5,thales,49300,<p>quelles sont les missions ?</p><p>qui somme...,M1802,CDI,quelles sont les missions ? qui sommes-nous ? ...,"à cholet, thales conçoit, développe, intègre, ..."
1625,REGIONSJOB_SUDOUEST_2021-10-14_3,2021,10,thales,31100.0,qui sommes-nous ? acteur spatial mondialement...,M1802,CDI,qui sommes-nous ? acteur spatial mondialement ...,acteur spatial mondialement reconnu dans les d...
1749,REGIONSJOB_OUEST_2022-02-08_3187,2022,2,thales,44300.0,qui sommes-nous ? thales propose des systèmes...,M1805,CDI,qui sommes-nous ? thales propose des systèmes ...,nos équipes de l'activité systèmes d'informati...
3446,jobintree_2023-06-05_11451,2023,6,thales,94220,description du poste :qui sommes-nous ?thales ...,H2502,CDI,description du poste :qui sommes-nous ?thales ...,des équipements aux systèmes en passant par le...
3570,api_poleemploi_2024-08-01_20653,2024,8,thales,None,qui sommes-nous ?\n face à la montée en pui...,M1805,CDI,qui sommes-nous ? face à la montée en puissanc...,face à la montée en puissance de la cybercrimi...
...,...,...,...,...,...,...,...,...,...,...
78819,meteojob_2022-05-10_12816,2022,5,thales,92230.0,qui sommes-nous ? thales propose des systèmes ...,M1802,CDI,qui sommes-nous ? thales propose des systèmes ...,des équipements aux systèmes en passant par le...
79656,hellowork_commercial_professionnel_2023-06-09_...,2023,6,thales,92230.0,qui sommes-nous ? thales propose des système...,M1605,Alternance,qui sommes-nous ? thales propose des systèmes ...,le site de gennevilliers est le coeur des acti...
79785,JOBINTREE_2021-05-26_6327,2021,5,thales,92230.0,description du poste :ce que nous pouvons acco...,H1206,None,description du poste :ce que nous pouvons acco...,description du poste :ce que nous pouvons acco...
80056,VIADEO_2019-02-27_3788,2019,2,thales,94150.0,ce que nous pouvons accomplir ensemble :vous s...,H1206,CDI,ce que nous pouvons accomplir ensemble :vous s...,ce que nous pouvons accomplir ensemble :vous s...


## Tests ex-post

In [ ]:
df["entreprise_nom"].value_counts().head(20)

In [ ]:
df_thales = df_small.loc[df_small["entreprise_nom"] == "thales"]

In [ ]:
df_thales.iloc[0]["description_clean"]

"à cholet, thales conçoit, développe, intègre, qualifie, industrialise, produit, déploie et maintient en condition opérationnelle des équipements de radiocommunications et des réseaux tactiques militaires, des équipements et systèmes de guerre électronique ainsi que des solutions de sécurité des systèmes d'informations (cryptage, réseaux, etc). c'est également à cholet que thales intègre, industrialise et produit les stations sol des réseaux de communication par satellites. * de formation ingénieur/bac+5, vous bénéficiez d'une expérience d'au moins 5 ans en ingénierie système, en ivvq ou en architecture électronique et/ou logicielle sur solutions complexes ? ce qu'on reconnaît chez vous : * votre aisance relationnelle et votre aptitude à travailler en équipe * votre force de proposition pour résoudre les problèmes techniques rencontrés * votre capacité à aller vers les autres pour capter, synthétiser et partager les informations vous souhaitez allier votre maîtrise du cycle de développ

In [ ]:
df_thales.iloc[0]["description"]

"<p>quelles sont les missions ?</p><p>qui sommes-nous ?<br/><br/> thales propose des systèmes d'information et de communication sécurisés et interopérables pour les forces armées, les forces de sécurité et les opérateurs d'importance vitale. ces activités, qui regroupent radiocommunications, réseaux, systèmes de protection, systèmes d'information critiques et cybersécurité, répondent aux besoins de marchés où l'utilisation des nouvelles technologies numériques est déterminante. thales intervient tout au long de la chaîne de valeur, des équipements aux systèmes en passant par le soutien logistique et les services associés.<br/><br/> à cholet, thales conçoit, développe, intègre, qualifie, industrialise, produit, déploie et maintient en condition opérationnelle des équipements de radiocommunications et des réseaux tactiques militaires, des équipements et systèmes de guerre électronique ainsi que des solutions de sécurité&nbsp;des systèmes d'informations (cryptage, réseaux, etc). c'est éga

In [ ]:
target = "créée en 1996"
firm = "okaidi"  # adapte



sub = df.loc[df["entreprise_nom"].astype("string").str.strip().str.lower().eq(firm), "description"].fillna("")
keys = []
for txt in sub.tolist():
    for s in split_sentences(txt):
        if target in s.lower():
            keys.append(sentence_key(s))



print("Nb occurrences trouvées:", len(keys))
print("Nb keys distinctes:", len(set(keys)))
print("Exemples keys:", list(set(keys))[:5])


Nb occurrences trouvées: 20
Nb keys distinctes: 6
Exemples keys: ['le groupe idkids compte plus de 0 0 magasins et 0 0 collaborateurs qui ont choisi de donner plus de sens a leur metier avec l engagement act for kids creee en 0', 'creee en 0 okaidi est aujourd hui une marque leader du pret a porter pour enfant de 0 a 0 ans', 'description du poste creee en 0 okaidi est aujourd hui une marque leader du pret a porter pour enfant de 0 a 0 ans marque universelle creative et respectueuse de l identite de chaque enfant okaidi propose des collections de vetements modernes faciles a porter et a mixer permettant a chacun d exprimer sa personnalite', 'l entreprise creee en 0 okaidi est aujourd hui une marque leader du pret a porter pour enfants de 0 a 0 ans', 'creee en 0 okaidi est aujourd hui une marque leader du pret a porter pour enfant de 0 a 0 ans avec plus de 0 magasins presents dans 0 pays dans le monde']


In [ ]:
firm = "okaidi"  # adapte exactement à ton entreprise_nom
sent = "créée en 1996, okaïdi est aujourd'hui une marque leader du prêt-à-porter pour enfant de 0 à 12 ans."

k_full = sentence_key(sent)  # key de la phrase entière

print("KEY(full):", k_full)
print("Is dropped?", k_full in drop_by_firm.get(firm, set()))

# Maintenant, regarde comment TON split découpe cette phrase
parts = split_sentences(sent)
print("\nSplit parts:")
for p in parts:
    print("-", p, " | key:", sentence_key(p), " | dropped:", sentence_key(p) in drop_by_firm.get(firm, set()))


KEY(full): creee en 0 okaidi est aujourd hui une marque leader du pret a porter pour enfant de 0 a 0 ans
Is dropped? True

Split parts:
- créée en 1996, okaïdi est aujourd'hui une marque leader du prêt-à-porter pour enfant de 0 à 12 ans.  | key: creee en 0 okaidi est aujourd hui une marque leader du pret a porter pour enfant de 0 a 0 ans  | dropped: True


In [ ]:
firm = "okaidi"
needle = "créée en 1996"

mask = df_small["entreprise_nom"].astype("string").str.strip().str.lower().eq(firm)
i = df_small.loc[mask & df_small["description"].astype("string").str.lower().str.contains(needle, na=False)].index[0]

print("BEFORE:\n", df_small.loc[i, "description"][:400], "\n")
print("AFTER:\n", df_small.loc[i, "description_clean"][:400])


BEFORE:
 créée en 1996, okaïdi est aujourd'hui une marque leader du prêt-à-porter pour enfant de 0 à 12 ans.  marque universelle, créative et respectueuse de l'identité de chaque enfant, okaïdi propose des collections de vêtements modernes, faciles à porter et à mixer permettant à chacun d'exprimer sa personnalité. au travers de messages optimistes et d'actions militantes, okaïdi encourage les enfants à êt 

AFTER:
 créée en 1996, okaïdi est aujourd'hui une marque leader du prêt-à-porter pour enfant de 0 à 12 ans. marque universelle, créative et respectueuse de l'identité de chaque enfant, okaïdi propose des collections de vêtements modernes, faciles à porter et à mixer permettant à chacun d'exprimer sa personnalité. au travers de messages optimistes et d'actions militantes, okaïdi encourage les enfants à êtr


In [ ]:
firm = "okaidi"
txt = df_small.loc[i, "description"]
drops = drop_by_firm.get(firm, set())

raw = clean_html_keep_newlines(txt).replace("\n", " ").strip()
sents = split_sentences(txt)

kept = []
removed = []
for sent in sents:
    k = sentence_key(sent)
    if k in drops:
        removed.append(sent)
    else:
        kept.append(sent)

out = _RE_WS.sub(" ", " ".join(kept)).strip()

print("removed_n:", len(removed))
print("out_len:", len(out), "raw_len:", len(raw))
print("fallback_triggered:", len(out) < 150)
print("\nREMOVED examples:", removed[:2])



removed_n: 34
out_len: 0 raw_len: 3283
fallback_triggered: True

REMOVED examples: ["créée en 1996, okaïdi est aujourd'hui une marque leader du prêt-à-porter pour enfant de 0 à 12 ans.", "marque universelle, créative et respectueuse de l'identité de chaque enfant, okaïdi propose des collections de vêtements modernes, faciles à porter et à mixer permettant à chacun d'exprimer sa personnalité."]


In [ ]:
print("firm cell repr:", repr(df_small.loc[i, "entreprise_nom"]))
print("firm normalized:", df_small.loc[i, "entreprise_nom"].strip().lower())
print("keys in drop_by_firm contain firm?", df_small.loc[i, "entreprise_nom"].strip() in drop_by_firm)


firm cell repr: 'okaidi'
firm normalized: okaidi
keys in drop_by_firm contain firm? True


# Embedding computation

Compute embeddings using a transformer

## Packages

In [ ]:
!pip install sentence-transformers torch

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 51.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.2/553.2 kB 44.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 68.6 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 95.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 65.7 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 59.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 79.5 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 66.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 82.7 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 88.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━

## Compute embeddings

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 312.41it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
texts = df["description_clean"].tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # cosine-friendly
)

df["embedding"] = list(embeddings)



Batches:  34%|███▍      | 4275/12579 [47:17<1:34:38,  1.46it/s]

In [ ]:
df.to_parquet("jocas_small/jocas_small_all_half_sirenised_embed.parquet")

## Compute PCA

In [ ]:
df = pd.read_parquet("jocas_small/jocas_small_all_half_sirenised_embed.parquet")

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

# X: (n_samples, n_dims) embeddings, e.g. numpy array
# If you have a list of vectors:
X = np.vstack(df["embedding"].to_numpy())

pca_full = PCA(svd_solver="full", random_state=0)
pca_full.fit(X)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
k90 = int(np.searchsorted(cumvar, 0.90) + 1)

print(f"Components for 90% variance: {k90}")
print(f"Cumulative variance at k90: {cumvar[k90-1]:.4f}")

# Now actually transform with that number of components
pca_90 = PCA(n_components=k90, svd_solver="full", random_state=0)
X_pca = pca_90.fit_transform(X)   # shape (n_samples, k90)

print("Original dim:", X.shape[1])
print("k for 90% variance:", k90)
print("Explained variance:", pca_90.explained_variance_ratio_.sum())


Components for 90% variance: 156
Cumulative variance at k90: 0.9013
Original dim: 384
k for 90% variance: 156
Explained variance: 0.9012588


In [ ]:
df["embedding_pca"] = list(X_pca.astype(np.float32))


,ID_JOCAS,annee,mois,entreprise_nom,location_zipcode,description,job_ROME_code,contractType,SIREN,NAF_Code,Employee_Range,Company_Category,Director_Birth_Year,Company_Creation_Date,description_clean,embedding
0,api_poleemploi_2023-04-28_78538,2023,4,None,31000,vous serez responsable du food truck pendant l...,G1401,CDI,,,,,,,vous serez responsable du food truck pendant l...,"[-0.037508223, 0.0065497477, -0.0007974197, -0..."
1,jobintree_2023-05-19_48068,2023,5,aunis sud,None,description du poste :avec des clients dans pl...,M1707,CDI,200041614,84.11Z,None,PME,0.0,2014-01-01,description du poste :avec des clients dans pl...,"[-0.06067792, 0.024671966, -0.024764137, -0.06..."
2,hellowork_bureau_etude_r_d_2023-03-17_1065,2023,3,None,None,vous êtes issu d'une formation supérieure typ...,,None,,,,,,,vous êtes issu d'une formation supérieure type...,"[-0.10698657, -0.06519823, -0.10766974, -0.079..."
3,jobintree_2023-06-07_33116,2023,6,win sport school,56000,"description du poste :win sport school vannes,...",M1705,Alternance,904549961,93.12Z,None,None,0.0,2021-10-13,description du poste :win sport school vannes ...,"[-0.010242393, 0.011192331, -0.062002324, -0.1..."
4,hellowork_logistique_metiers_transport_2023-03...,2023,3,assystem,1150.0,qualifications vous recherchez une alternance...,N1301,CDI,412076937,70.10Z,None,ETI,1968,1997-04-26,qualifications vous recherchez une alternance ...,"[-0.09042679, -0.088197626, -0.04862888, -0.04..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
811174,JOBINTREE_2020-12-05_8970,2020,12,lustral,39100.0,"nous recherchons une gouvernant d'intérieur, m...",K1304,CDI,,,,,,,"nous recherchons une gouvernant d'intérieur, m...","[-0.063871354, -0.0012015791, -0.036611278, -0..."
811175,JOBINTREE_2020-01-24_77816,2020,1,la pie verte,31700.0,préparation de commandes et livraisons de repa...,N4105,CDD,,,,,,,préparation de commandes et livraisons de repa...,"[-0.07240109, 0.0437684, -0.037568178, -0.0340..."
811176,JOBINTREE_2020-02-20_19748,2020,2,jvs mairistem,51520.0,tes missions : au sein de notre service commer...,D1401,CDI,,,,,,,tes missions : au sein de notre service commer...,"[-0.099452265, -0.014010274, -0.037381187, -0...."
811177,API_POLEEMPLOI_2020-10-28_4237,2020,10,None,49700.0,description du poste :\nvous mettez en œuvre d...,F1703,MIS,,,,,,,description du poste : vous mettez en œuvre de...,"[-0.065282054, 0.10251117, -0.017559458, -0.05..."


In [ ]:
df.to_parquet("jocas_small/jocas_small_all_half_sirenised_embed.parquet")